# Thermal-Ops with GRPO using TRL

Train an agentic model to manage Data Center Cooling.


In [1]:
!nvidia-smi
!pip install trackio
%pip install --no-cache-dir auto-gptq optimum --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu121/
!pip install ipywidgets

/bin/bash: line 1: nvidia-smi: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 18.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 120.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
Looking in indexes: https://pypi.org/simple, https://huggingface.github.io/autogptq-index/whl/cu121/
     ━━━━━━━━━━

In [6]:
pip install optimum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 4.0 MB/s eta 0:00:00a 0:00:01


### Log in to Hugging Face


In [3]:
from huggingface_hub import notebook_login

notebook_login()


## Define the system prompt


In [4]:
prompt = prompt = """You are an expert Data Center Facility Manager.
You manage 3 server racks that must stay between 20.0°C and 25.0°C.
## Available Tools
Call tools using EXACTLY this format (one per message):
<tool_call>{"name": "set_fan_speed", "arguments": {"rack_id": 0, "rpm": 2000}}</tool_call>
<tool_call>{"name": "adjust_chiller", "arguments": {"chiller_temp": 18.0}}</tool_call>
<tool_call>{"name": "migrate_workload", "arguments": {"source_rack": 0, "target_rack": 1}}</tool_call>
<tool_call>{"name": "wait", "arguments": {}}</tool_call>
## Rules
- GREEN (< 25°C): Optimal — minimal energy cost
- YELLOW (25-27°C): Warning — throttling penalty
- RED (> 27°C): Critical — hardware damage penalty
- Max fan speed: 5000 RPM
- Broken fans cannot be used
- Minimise energy while keeping temps safe
Always respond with exactly ONE tool call. No explanation needed."""


## Define the environment
The `ThermalOpsEnv` class implements all the thermodynamics simulation explicitly inside the environment.


In [21]:
import json
import torch
import optimum
import random

class ThermalOpsEnv:
    def __init__(self):
        self.num_racks = 3
        self.max_steps = 10
        self.w1_energy = 0.5
        self.w2_overheat = 1000.0
        self.safe_temp_max = 25.0
        self.critical_temp = 27.0
        
        self.reset()
        
    def reset(self, **kwargs) -> str:
        self.step_count = 0
        
        # ← NEW: Randomised starting conditions
        self.ambient_temp = random.uniform(20.0, 28.0)
        self.rack_temps = [
            random.uniform(20.0, 26.0) for _ in range(self.num_racks)
        ]
        self.power_loads = [
            random.uniform(5.0, 20.0) for _ in range(self.num_racks)
        ]
        self.fan_rpms = [
            random.choice([500, 1000, 1500, 2000]) for _ in range(self.num_racks)
        ]
        self.chiller_setpoint = random.uniform(10.0, 20.0)
        self.energy_cost = random.uniform(0.10, 0.25)
        
        self.total_energy_consumed = 0.0
        
        # ← NEW: Random hardware failures (30% chance per fan)
        self.broken_fans = set(
            i for i in range(self.num_racks) if random.random() < 0.3
        )
        
        self.reward = 0.0
        self.done = False
        
        return self._get_obs()
        
    def _get_obs(self) -> str:
        obs = {
            "ambient_temp": self.ambient_temp,
            "rack_temps": [round(t, 2) for t in self.rack_temps],
            "power_loads": [round(l, 2) for l in self.power_loads],
            "fan_rpms": self.fan_rpms,
            "chiller_setpoint": round(self.chiller_setpoint, 2),
            "broken_fans": list(self.broken_fans),
            "step_count": self.step_count
        }
        return f"Observation: {json.dumps(obs)}"

    def set_fan_speed(self, rack_id: int, rpm: int) -> str:
        """
        Set the fan speed for a specific server rack.

        Args:
            rack_id: The ID of the server rack (0, 1, or 2).
            rpm: The speed to set the fan to in RPM (0 to 5000). High RPM cools faster but uses exponential power.
            
        Returns:
            Status message of the action.
        """
        if self.done: return "Episode over."
        if 0 <= rack_id < self.num_racks and rack_id not in self.broken_fans:
            self.fan_rpms[rack_id] = max(0, min(5000, rpm))
            return f"Fan {rack_id} speed bounded and set to {self.fan_rpms[rack_id]} RPM."
        return "Failed: Invalid rack_id or fan is broken."

    def adjust_chiller(self, chiller_temp: float) -> str:
        """
        Adjust the ambient liquid chiller base temperature setpoint.

        Args:
            chiller_temp: The temperature to set the chiller to (in Celsius).
            
        Returns:
            Status message.
        """
        if self.done: return "Episode over."
        self.chiller_setpoint = max(5.0, min(30.0, float(chiller_temp)))
        return f"Chiller setpoint adjusted to {self.chiller_setpoint}°C."

    def migrate_workload(self, source_rack: int, target_rack: int) -> str:
        """
        Migrate power load (heat generation) from one rack to another.

        Args:
            source_rack: The rack ID to pull workload from.
            target_rack: The rack ID to push workload to.
            
        Returns:
            Status message.
        """
        if self.done: return "Episode over."
        if 0 <= source_rack < self.num_racks and 0 <= target_rack < self.num_racks and source_rack != target_rack:
            load_to_move = self.power_loads[source_rack] * 0.5
            self.power_loads[source_rack] -= load_to_move
            self.power_loads[target_rack] += load_to_move
            return f"Migrated {load_to_move:.2f} workload from rack {source_rack} to rack {target_rack}."
        return "Failed: Invalid source or target rack."
        
    def wait(self) -> str:
        """
        Wait and progress the environment thermodynamics by 1 simulation step.

        Returns:
            The new Observation of the environment after the step.
        """
        if self.done:
            raise ValueError("Environment episode is finished.")
            
        energy_consumed = 0.0
        overheat_penalty_total = 0.0
        
        chiller_delta = max(0, self.ambient_temp - self.chiller_setpoint)
        energy_consumed += 0.5 * (chiller_delta ** 2)
        
        for i in range(self.num_racks):
            heat_generated = 0.1 * self.power_loads[i]
            rpm = self.fan_rpms[i] if i not in self.broken_fans else 0
            cooling_power = (rpm / 1000.0) * 0.5
            chiller_effect = max(0, self.rack_temps[i] - self.chiller_setpoint) * 0.1
            ambient_effect = (self.ambient_temp - self.rack_temps[i]) * 0.05
            
            self.rack_temps[i] += heat_generated - cooling_power - chiller_effect + ambient_effect
            energy_consumed += ((rpm / 1000.0) ** 3) * 0.2

            # Gently penalize drifting away from the perfect 22.0C baseline
            baseline_drift_penalty = abs(self.rack_temps[i] - 22.0) * 0.05
            overheat_penalty_total += baseline_drift_penalty
            
            if self.rack_temps[i] > self.safe_temp_max:
                if self.rack_temps[i] > self.critical_temp:
                    overheat_penalty_total += self.w2_overheat
                else:
                    overheat_penalty_total += self.w2_overheat * 0.1 * (self.rack_temps[i] - self.safe_temp_max)
                    
        cost = energy_consumed * self.energy_cost
        self.total_energy_consumed += energy_consumed
        
        # Calculate step reward
        self.reward += -(self.w1_energy * cost) - overheat_penalty_total
        
        self.step_count += 1
        if self.step_count >= self.max_steps:
            self.done = True
            
        return self._get_obs()



## Define the reward function
The reward function retrieves the final `env.reward` score.


In [8]:
def reward_func(environments, completions, **kwargs) -> list[float]:
    rewards = []
    for env, completion in zip(environments, completions):
        r = 0.0

        # 1. Base environment reward (energy + overheat penalty)
        r += env.reward

        # 2. Tool-use format bonus: did the model actually try to call a tool?
        text = completion  # or however TRL passes the completion string
        if isinstance(text, list):
            text = " ".join([m.get("content", "") for m in text])

        import re
        tool_calls = re.findall(r'"(set_fan_speed|adjust_chiller|migrate_workload|wait)"', text)
        r += len(tool_calls) * 5.0  # reward each valid tool call attempt

        # 3. JSON format bonus: reward well-formed JSON
        json_blocks = re.findall(r'\{[^}]+\}', text)
        valid_json_count = 0
        for block in json_blocks:
            try:
                json.loads(block)
                valid_json_count += 1
            except:
                pass
        r += valid_json_count * 3.0

        # 4. Penalise pure-text hallucination (no tool calls at all)
        if len(tool_calls) == 0:
            r -= 10.0

        rewards.append(r)
    return rewards


## Create Dataset for Rollouts


In [11]:
%pip uninstall -y Dataset dataset
%pip install -U sqlalchemy datasets

Found existing installation: dataset 1.6.2
Uninstalling dataset-1.6.2:
  Successfully uninstalled dataset-1.6.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 35.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 1.4.54
    Uninstalling SQLAlchemy-1.4.54:
      Successfully uninstalled SQLAlchemy-1.4.54
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [15]:
import random
from datasets import Dataset
def make_scenario_prompt(idx):
    """Generate a unique scenario for each training example."""
    # Vary the difficulty
    scenarios = [
        "All systems are nominal. Optimise for minimum energy cost.",
        "WARNING: Fan 2 has failed! Rack 2 temperature is rising rapidly.",
        "Power load spike detected on Rack 0 — a compute job just started.",
        "External ambient temperature has risen to 30°C due to a heatwave.",
        "Budget alert: Energy costs have doubled. Minimise consumption.",
        "Rack 1 is approaching critical temperature (26.5°C). Act fast.",
        "Night mode: all workloads are low. Reduce cooling to save energy.",
        "A new workload batch is about to start on Rack 0 and Rack 2.",
    ]
    scenario = scenarios[idx % len(scenarios)]
    
    full_prompt = f"""{prompt}

Current Scenario: {scenario}

Respond ONLY with tool calls in this exact format:
<tool_call>{{"name": "tool_name", "arguments": {{"arg": value}}}}</tool_call>
"""
    return [{"role": "user", "content": full_prompt}]

dataset = Dataset.from_dict({
    "prompt": [make_scenario_prompt(i) for i in range(500)]
})



## Set GRPO Config


In [16]:
# 1. Install/Update bitsandbytes and accelerate (required for 4-bit)
%pip install -U bitsandbytes>=0.46.1 accelerate transformers trl datasets

# 2. Re-verify the install
import bitsandbytes as bnb
import torch
print(f"Bitsandbytes version: {bnb.__version__}")
print(f"Is CUDA available for bnb? {torch.cuda.is_available()}")

Bitsandbytes version: 0.49.2
Is CUDA available for bnb? False


In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from trl.chat_template_utils import qwen3_schema

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Load in safe FP32 to prevent scaler crashes
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float32, 
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.response_schema = qwen3_schema

# 2. Config WITH Hugging Face Space Integration
grpo_config = GRPOConfig(
    num_train_epochs=3,              # ← was 1: more passes over the data
    learning_rate=5e-5,              # ← was 1e-5: higher LR for small model
    lr_scheduler_type="cosine",      # ← add: smooth LR decay
    warmup_ratio=0.1,               # ← add: warm up to avoid early instability
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,   # ← was 16: faster iteration
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    
    num_generations=4,               # ← was 2: CRITICAL — more diversity for GRPO
    max_completion_length=512,       # ← was 256: room for multi-tool chains
    gradient_checkpointing=True,
    use_vllm=False,
    
     # --- Hugging Face Integration Restored ---
    output_dir="thermal-ops-0.5B",
    report_to="trackio",
    trackio_space_id="thermal-ops-0.5B",
    push_to_hub=True,
    logging_steps=1,
    save_steps=20,                   # ← was 10: less I/O overhead
    save_total_limit=2,              # ← keep best 2 checkpoints
    
    # GRPO-specific
    max_grad_norm=0.5,                
    
   

)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Create Trainer


In [23]:
pip install jmespath

In [24]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=ThermalOpsEnv,
)

/tmp/ipykernel_8932/3703186129.py:1: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(


In [21]:
%pip install jmespath

## Memory Stats & Training Loop


In [25]:
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


* Trackio project initialized: huggingface
* Trackio metrics will be synced to Hugging Face Dataset: VECTOR2356/thermal-ops-0.5B-dataset
* Creating new space: https://huggingface.co/spaces/VECTOR2356/thermal-ops-0.5B
* Waiting for Space to be ready...
* View dashboard by going to: https://VECTOR2356-thermal-ops-0.5B.hf.space/


* NVIDIA GPU detected, enabling automatic GPU metrics logging
* Created new run: VECTOR2356-1774872869


Step,Training Loss
1,-0.038743
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

* Run finished. Uploading logs to Trackio Space (please wait...)


## Evaluate Inference
Run the fine-tuned model against a generic Thermal-Ops environment test.


In [27]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import re

# Make sure this matches the output_dir from your GRPOConfig
output_dir = "thermal-ops-0.5B" 

# Load the fine-tuned model and its tokenizer natively
fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    output_dir, 
    torch_dtype=torch.float32, 
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(output_dir)

def play_thermal_ops(model, tokenizer):
    env = ThermalOpsEnv()
    
    # Introduce random difficulty (Hardware Failure Scenario)
    env.broken_fans.add(2)
    obs = env.reset()

    print("Initial observation:")
    print(obs)
    print()

    messages = [{"role": "user", "content": prompt}, {"role": "user", "content": obs}]

    for turn in range(env.max_steps * 5): # Maximum 5 tool calls per real step
        if env.done:
            break

        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False, # Make sure this matches your Qwen template
        )
        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        
        # We don't need disable_compile anymore, removing to prevent the deprecation warning
        generated_ids = model.generate(**model_inputs, max_new_tokens=256)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)

        print(f"Model output: {generated_text}")
        
        try:
            # Parse tool
            if "{" in generated_text and "}" in generated_text:
                start = generated_text.index("{")
                end = generated_text.rindex("}") + 1
                args = json.loads(generated_text[start:end])
                
                tool_name = "wait"
                if "set_fan_speed" in generated_text: tool_name = "set_fan_speed"
                elif "adjust_chiller" in generated_text: tool_name = "adjust_chiller"
                elif "migrate_workload" in generated_text: tool_name = "migrate_workload"

                # Execute
                if tool_name == "set_fan_speed":
                    feedback = env.set_fan_speed(args.get("rack_id", 0), args.get("rpm", 1000))
                elif tool_name == "adjust_chiller":
                    feedback = env.adjust_chiller(args.get("chiller_temp", 20.0))
                elif tool_name == "migrate_workload":
                    feedback = env.migrate_workload(args.get("source_rack", 0), args.get("target_rack", 1))
                else:
                    feedback = env.wait()
                    
                print(f"    Feedback: {feedback.strip()}")
                
                messages.append({"role": "assistant", "content": generated_text})
                messages.append({"role": "user", "content": feedback})
            else:
                feedback = env.wait()
                messages.append({"role": "assistant", "content": generated_text})
                messages.append({"role": "user", "content": "Format error. Skipped: " + feedback})
                
        except Exception as e:
            print(f"Error executing Action: {e}")
            break

    print(f"Game finished! Final reward computed sum: {env.reward}")

# Run the simulation
play_thermal_ops(fine_tuned_model, tokenizer)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Initial observation:
Observation: {"ambient_temp": 22.0, "rack_temps": [22.0, 22.0, 22.0], "power_loads": [10.0, 10.0, 10.0], "fan_rpms": [1000, 1000, 1000], "chiller_setpoint": 15.0, "step_count": 0}

Model output: Based on the observed data, we can identify several key factors contributing to the current temperature profile and potential issues in the thermal system. Let's analyze the data to determine if the current operating conditions are within the desired range of 22°C to 27°C, which is the critical temperature for most servers, typically around 27°C, but it is important to note that this is not an absolute limit and depends on other environmental factors such as humidity, air density, and specific heat capacity of the surrounding air.

### 1. **Cooling Efficiency**
The observed cooling efficiency is 15% with a fan rpm of 1000 RPM, indicating that there is still room for improvement. A higher rpm will increase the overall cooling capacity, but it also increases the power consump